# 🧪 Vanilla Transformer Baseline for ARC-AGI-1

**Architecture:**
- 13 Transformer Encoder blocks
- Model dim = 128
- FFN: 4 hidden layers, hidden dim = 2048
- Multi-head attention (8 heads) + residual connections + LayerNorm
- Dropout throughout

**Training:** Encoder training on Re-ARC (input grid → output grid prediction)  
**Evaluation:** ARC-AGI-1 benchmark (400 evaluation tasks)

In [ ]:
import sys, os, subprocess

KAGGLE_REPO_PATH = '/kaggle/working/Model-Jepa'
if os.path.exists(KAGGLE_REPO_PATH):
    print('✅ Kaggle detected. Pulling latest...')
    subprocess.run(f'cd {KAGGLE_REPO_PATH} && git pull origin main || true', shell=True, check=True)
    if KAGGLE_REPO_PATH not in sys.path:
        sys.path.insert(0, KAGGLE_REPO_PATH)
else:
    sys.path.insert(0, os.path.abspath('.'))
    print('✅ Local mode.')

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import json, pathlib, random, math, time
from collections import defaultdict

if torch.cuda.is_available(): DEVICE = 'cuda'
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available(): DEVICE = 'mps'
else: DEVICE = 'cpu'
print(f'Device: {DEVICE}')
print(f'PyTorch: {torch.__version__}')

## 1. Data Loading

- **Training:** Re-ARC (procedurally generated, ~40k pairs)
- **Evaluation:** ARC-AGI-1 evaluation set (400 tasks, ~1200 test pairs)

In [ ]:
MAX_GRID = 30
NUM_COLORS = 11  # 0-9 colors + 1 padding token
PAD_TOKEN = 10

def pad_grid(grid, max_size=MAX_GRID):
    """Pad a 2D list grid to max_size×max_size, using PAD_TOKEN for padding."""
    arr = np.array(grid, dtype=np.int64)
    h, w = arr.shape
    out = np.full((max_size, max_size), PAD_TOKEN, dtype=np.int64)
    out[:min(h, max_size), :min(w, max_size)] = arr[:min(h, max_size), :min(w, max_size)]
    return out

def load_rearc(data_path, max_pairs_per_task=100):
    """Load Re-ARC dataset as (input_grid, output_grid) pairs."""
    path = pathlib.Path(data_path)
    pairs = []
    task_files = sorted(path.rglob('*.json'))
    for fpath in task_files:
        try:
            examples = json.loads(fpath.read_text())
            for ex in examples[:max_pairs_per_task]:
                inp = pad_grid(ex['input'])
                out = pad_grid(ex['output'])
                pairs.append((inp, out))
        except Exception:
            pass
    print(f'Re-ARC: loaded {len(pairs):,} pairs from {len(task_files)} tasks')
    return pairs

def load_arc_eval(data_path):
    """Load ARC-AGI-1 evaluation tasks. Returns list of tasks, each with train+test pairs."""
    path = pathlib.Path(data_path)
    tasks = []
    for fpath in sorted(path.glob('*.json')):
        try:
            task = json.loads(fpath.read_text())
            tasks.append({'id': fpath.stem, 'train': task['train'], 'test': task['test']})
        except Exception:
            pass
    print(f'ARC Eval: loaded {len(tasks)} tasks')
    return tasks

# ─── Load datasets ───
REARC_PATH = '/kaggle/working/Model-Jepa/arc_data/re-arc'
ARC_EVAL_PATH = '/kaggle/working/arc-agi_evaluation-challenges-20240612/arc-agi_evaluation_challenges'

# Fallback paths for local dev
if not os.path.exists(REARC_PATH):
    REARC_PATH = 'arc_data/re-arc'
if not os.path.exists(ARC_EVAL_PATH):
    ARC_EVAL_PATH = 'arc_data/evaluation'  # adjust to your local path

train_pairs = load_rearc(REARC_PATH)

# Shuffle and split
random.seed(42)
random.shuffle(train_pairs)
val_size = int(len(train_pairs) * 0.05)
val_pairs = train_pairs[:val_size]
train_pairs = train_pairs[val_size:]
print(f'Train: {len(train_pairs):,} | Val: {len(val_pairs):,}')

## 2. Model Architecture

Pure Transformer Encoder with:
- Learned token embeddings (11 colors: 0-9 + pad)
- 2D positional encoding
- 13 Transformer blocks (attention + 4-layer FFN + residual + LayerNorm)
- Output head: per-token classification (predict output color at each grid position)

In [ ]:
class DeepFFN(nn.Module):
    """4-layer FFN with hidden dim 2048, as specified."""
    def __init__(self, d_model, d_hidden=2048, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_hidden, d_hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_hidden, d_hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_hidden, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    """Standard Transformer block: LayerNorm → MHA → Residual → LayerNorm → FFN → Residual."""
    def __init__(self, d_model=128, n_heads=8, d_ffn=2048, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.drop1 = nn.Dropout(dropout)

        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = DeepFFN(d_model, d_ffn, dropout)
        self.drop2 = nn.Dropout(dropout)

    def forward(self, x):
        # Pre-norm architecture
        h = self.norm1(x)
        h, _ = self.attn(h, h, h)
        x = x + self.drop1(h)      # Residual connection

        h = self.norm2(x)
        h = self.ffn(h)
        x = x + self.drop2(h)      # Residual connection
        return x


class ARCTransformer(nn.Module):
    """
    Vanilla Transformer encoder for ARC grid-to-grid prediction.

    Input:  30×30 grid of color tokens (0-9, 10=pad)
    Output: 30×30 grid of predicted output color logits

    The input grid is flattened to 900 tokens. Each token gets:
      - color embedding (learned, dim=128)
      - 2D positional embedding (learned, dim=128)
    """
    def __init__(self, d_model=128, n_heads=8, n_layers=13, d_ffn=2048,
                 dropout=0.1, max_grid=30, n_colors=11):
        super().__init__()
        self.d_model = d_model
        self.max_grid = max_grid
        self.n_colors = n_colors

        # Token embedding: color → d_model
        self.color_embed = nn.Embedding(n_colors, d_model)

        # 2D positional embeddings (row + col)
        self.row_embed = nn.Embedding(max_grid, d_model)
        self.col_embed = nn.Embedding(max_grid, d_model)

        # Input projection (combine color + position)
        self.input_proj = nn.Linear(d_model, d_model)
        self.input_drop = nn.Dropout(dropout)

        # 13 Transformer blocks
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ffn, dropout)
            for _ in range(n_layers)
        ])

        # Final norm + output head
        self.final_norm = nn.LayerNorm(d_model)
        self.output_head = nn.Linear(d_model, n_colors)  # per-token color prediction

        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def forward(self, grid):
        """
        Args:
            grid: [B, 30, 30] LongTensor of color indices (0-10)
        Returns:
            logits: [B, 30, 30, n_colors] prediction for each position
        """
        B, H, W = grid.shape

        # Color embeddings
        tok = self.color_embed(grid)  # [B, H, W, d_model]

        # Position embeddings
        rows = torch.arange(H, device=grid.device)
        cols = torch.arange(W, device=grid.device)
        pos = self.row_embed(rows).unsqueeze(1) + self.col_embed(cols).unsqueeze(0)  # [H, W, d]

        # Combine
        x = tok + pos.unsqueeze(0)   # [B, H, W, d_model]
        x = self.input_proj(x)
        x = self.input_drop(x)

        # Flatten spatial dims: [B, H*W, d_model]
        x = x.view(B, H * W, self.d_model)

        # Transformer blocks
        for block in self.blocks:
            x = block(x)

        # Output
        x = self.final_norm(x)
        logits = self.output_head(x)  # [B, H*W, n_colors]
        logits = logits.view(B, H, W, self.n_colors)

        return logits


# Build model
model = ARCTransformer(
    d_model=128,
    n_heads=8,
    n_layers=13,
    d_ffn=2048,
    dropout=0.1,
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {total_params:,}')
print(f'Architecture: 13 blocks × (8-head attn + 4-layer FFN[128→2048→2048→2048→128])')

## 3. Training Loop

Encoder training: Input grid → Transformer → Predict output grid at every position.

In [ ]:
import wandb
wandb.login(key=os.environ.get('WANDB_API_KEY', ''))

# ─── Training config ───
EPOCHS = 50
BATCH_SIZE = 32
LR = 3e-4
WARMUP_STEPS = 500
GRAD_CLIP = 1.0

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)

# Warmup + cosine decay scheduler
total_steps = EPOCHS * (len(train_pairs) // BATCH_SIZE)
def lr_lambda(step):
    if step < WARMUP_STEPS:
        return step / WARMUP_STEPS
    progress = (step - WARMUP_STEPS) / max(1, total_steps - WARMUP_STEPS)
    return 0.5 * (1 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

wb_run = wandb.init(project='NS-ARC-Slotted', name='TransformerBaseline-13L', reinit=True,
                    config={'d_model': 128, 'n_layers': 13, 'd_ffn': 2048,
                            'dropout': 0.1, 'batch_size': BATCH_SIZE, 'lr': LR})


def make_batch(pairs, batch_size):
    """Sample a batch from pair list."""
    chosen = random.sample(pairs, min(batch_size, len(pairs)))
    inp = torch.tensor(np.stack([p[0] for p in chosen]), dtype=torch.long, device=DEVICE)  # [B, 30, 30]
    tgt = torch.tensor(np.stack([p[1] for p in chosen]), dtype=torch.long, device=DEVICE)  # [B, 30, 30]
    return inp, tgt


# ─── Training ───
print(f'Training: {EPOCHS} epochs, {len(train_pairs)//BATCH_SIZE} batches/epoch')
global_step = 0
best_val_acc = 0.0

for epoch in range(EPOCHS):
    model.train()
    t0 = time.time()
    epoch_loss = 0.0
    epoch_correct = 0
    epoch_total = 0
    n_batches = len(train_pairs) // BATCH_SIZE

    for batch_idx in range(n_batches):
        inp, tgt = make_batch(train_pairs, BATCH_SIZE)

        logits = model(inp)  # [B, 30, 30, 11]

        # Cross-entropy loss (ignore padding token)
        loss = F.cross_entropy(
            logits.reshape(-1, NUM_COLORS),
            tgt.reshape(-1),
            ignore_index=PAD_TOKEN
        )

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        scheduler.step()
        global_step += 1

        # Accuracy (non-pad tokens only)
        with torch.no_grad():
            preds = logits.argmax(dim=-1)      # [B, 30, 30]
            mask = tgt != PAD_TOKEN
            correct = (preds == tgt) & mask
            epoch_correct += correct.sum().item()
            epoch_total += mask.sum().item()
            epoch_loss += loss.item()

        if global_step % 100 == 0:
            wb_run.log({
                'train/loss': loss.item(),
                'train/lr': optimizer.param_groups[0]['lr'],
                'train/step': global_step,
            })

    # ─── Epoch stats ───
    avg_loss = epoch_loss / n_batches
    train_acc = epoch_correct / max(1, epoch_total)
    elapsed = time.time() - t0

    # ─── Validation ───
    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for vi in range(0, len(val_pairs), BATCH_SIZE):
            batch = val_pairs[vi:vi+BATCH_SIZE]
            if not batch: break
            inp = torch.tensor(np.stack([p[0] for p in batch]), dtype=torch.long, device=DEVICE)
            tgt = torch.tensor(np.stack([p[1] for p in batch]), dtype=torch.long, device=DEVICE)
            logits = model(inp)
            preds = logits.argmax(dim=-1)
            mask = tgt != PAD_TOKEN
            val_correct += ((preds == tgt) & mask).sum().item()
            val_total += mask.sum().item()

    val_acc = val_correct / max(1, val_total)

    wb_run.log({
        'epoch': epoch,
        'train/epoch_loss': avg_loss,
        'train/epoch_acc': train_acc,
        'val/acc': val_acc,
    })

    print(f'[Epoch {epoch:3d}] Loss: {avg_loss:.4f}  Train_Acc: {train_acc:.4f}  Val_Acc: {val_acc:.4f}  ({elapsed:.1f}s)')

    # Save best
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'checkpoints/transformer_baseline_best.pt')
        print(f'  ✅ New best val acc: {val_acc:.4f}')

print(f'\n✅ Training complete. Best Val Acc: {best_val_acc:.4f}')

## 4. ARC-AGI-1 Evaluation

For each ARC evaluation task:
1. Show the model the test input grid
2. Predict the output grid
3. Compare to ground truth

Metrics:
- **Per-pixel accuracy**: fraction of correctly predicted non-pad pixels
- **Perfect grid accuracy**: fraction of tasks where the ENTIRE output grid is predicted correctly

In [ ]:
os.makedirs('checkpoints', exist_ok=True)

# Load best model
if os.path.exists('checkpoints/transformer_baseline_best.pt'):
    model.load_state_dict(torch.load('checkpoints/transformer_baseline_best.pt', map_location=DEVICE))
    print('✅ Loaded best checkpoint')

model.eval()

# Load ARC eval tasks
if os.path.exists(ARC_EVAL_PATH):
    eval_tasks = load_arc_eval(ARC_EVAL_PATH)
else:
    print(f'⚠️ ARC eval path not found: {ARC_EVAL_PATH}')
    print('Skipping ARC evaluation. To run evaluation, provide the path to ARC-AGI-1 evaluation JSONs.')
    eval_tasks = []


def evaluate_arc(model, tasks, device):
    """Evaluate on ARC-AGI-1 tasks."""
    total_pixel_correct = 0
    total_pixel_count = 0
    perfect_grids = 0
    total_grids = 0
    results = []

    with torch.no_grad():
        for task in tasks:
            task_id = task['id']

            for test_pair in task['test']:
                inp_raw = test_pair['input']
                out_raw = test_pair['output']

                # Get actual dimensions of the output
                out_h, out_w = len(out_raw), len(out_raw[0])

                # Pad and predict
                inp_padded = pad_grid(inp_raw)   # [30, 30]
                out_padded = pad_grid(out_raw)   # [30, 30]

                inp_t = torch.tensor(inp_padded, dtype=torch.long, device=device).unsqueeze(0)  # [1, 30, 30]
                logits = model(inp_t)             # [1, 30, 30, 11]
                pred = logits.argmax(dim=-1)[0]   # [30, 30]

                # Compare only within actual output dimensions
                tgt = torch.tensor(out_padded, dtype=torch.long, device=device)
                mask = tgt != PAD_TOKEN

                correct = (pred == tgt) & mask
                n_correct = correct.sum().item()
                n_total = mask.sum().item()

                total_pixel_correct += n_correct
                total_pixel_count += n_total

                # Perfect grid: check only within actual output bounds
                pred_crop = pred[:out_h, :out_w].cpu().numpy()
                tgt_crop = np.array(out_raw, dtype=np.int64)
                is_perfect = np.array_equal(pred_crop, tgt_crop)
                if is_perfect:
                    perfect_grids += 1
                total_grids += 1

                results.append({
                    'task_id': task_id,
                    'pixel_acc': n_correct / max(1, n_total),
                    'perfect': is_perfect,
                })

    overall_pixel_acc = total_pixel_correct / max(1, total_pixel_count)
    overall_perfect = perfect_grids / max(1, total_grids)

    return {
        'pixel_accuracy': overall_pixel_acc,
        'perfect_grid_accuracy': overall_perfect,
        'perfect_grids': perfect_grids,
        'total_grids': total_grids,
        'per_task': results,
    }


if eval_tasks:
    print(f'\n🔍 Evaluating on {len(eval_tasks)} ARC-AGI-1 tasks...')
    results = evaluate_arc(model, eval_tasks, DEVICE)

    print(f'\n{"="*60}')
    print(f'ARC-AGI-1 Evaluation Results')
    print(f'{"="*60}')
    print(f'Per-pixel accuracy:      {results["pixel_accuracy"]*100:.2f}%')
    print(f'Perfect grid accuracy:   {results["perfect_grid_accuracy"]*100:.2f}%')
    print(f'Perfect grids:           {results["perfect_grids"]} / {results["total_grids"]}')
    print(f'{"="*60}')

    wb_run.log({
        'eval/pixel_accuracy': results['pixel_accuracy'],
        'eval/perfect_grid_accuracy': results['perfect_grid_accuracy'],
        'eval/perfect_grids': results['perfect_grids'],
    })

    # Show top-performing tasks
    perfect_tasks = [r for r in results['per_task'] if r['perfect']]
    if perfect_tasks:
        print(f'\n✅ Perfectly solved tasks ({len(perfect_tasks)}): ')
        for r in perfect_tasks[:20]:
            print(f'  {r["task_id"]}')
else:
    print('Skipping evaluation (no eval tasks loaded).')

wb_run.finish()
print('\n✅ Experiment complete.')